# Kiểm định gold seed — đối chiếu phiếu LF với nhãn người

Notebook so sánh **riêng** (rule 8: không nhét so sánh vào notebook LF). Với mỗi cặp `(lf, task)` có trong phiếu, đo độ tin trên gold seed rồi áp cổng go/no-go.

Chạy CPU (pandas + sklearn), không GPU.

Định nghĩa `alpha`, `kappa`, `coverage`, cổng go/no-go: xem thêm tại `docs/LF1_Methodology.md` và `docs/LF3_Methodology.md` (mục "Cổng go/no-go"). Bộ nhãn & required-view: xem thêm tại `docs/Labeling_Plan.md`. Schema phiếu & `fuse_votes`: xem thêm tại `src/utils/lf_io.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import precision_recall_fscore_support


def find_root(start):
    marker = Path("gold_seed") / "gold_seed_labels.csv"
    candidates = []
    candidates.append(start)
    for parent in start.parents:
        candidates.append(parent)
    for cand in candidates:
        probe = cand / marker
        if probe.exists():
            return cand
    raise SystemExit("Khong thay gold_seed/gold_seed_labels.csv — chay notebook ben trong repo coconut-iqa")


ROOT = find_root(Path.cwd())

UTILS = ROOT / "src" / "utils" / "lf_io.ipynb"
get_ipython().run_line_magic("run", str(UTILS))

/sessions/serene-nice-babbage/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [2]:
# Duong dan dau vao/ra (hardcode, fail-fast, khong auto-discovery).
GOLD_CSV = ROOT / "gold_seed" / "gold_seed_labels.csv"
VOTES_DIR = ROOT / "labels" / "votes"
OOF_RAW_CSV = ROOT / "labels" / "disease_clf" / "oof_predictions.csv"
REPORT_CSV = ROOT / "labels" / "validation" / "gold_seed_report.csv"

# Cong go/no-go.
ALPHA_MIN = 0.5
COVERAGE_MIN = 0.5

# Luoi tau [0.00 .. 0.95] buoc 0.05.
TAU_GRID = []
_tau = 0.0
while _tau <= 0.95001:
    TAU_GRID.append(round(_tau, 2))
    _tau = _tau + 0.05

# Task ky vong cho tung file phieu (dung khi file chi co header -> khong doc duoc cot 'lf').
LF_TASK_HINT = {}
LF_TASK_HINT["lf1_maturity"] = "1_maturity_evaluation"
LF_TASK_HINT["lf2_foliar"] = "2_foliar_disease"
LF_TASK_HINT["lf3_trunk"] = "3_trunk_disease"
LF_TASK_HINT["lf4_crown"] = "4_crown_disease"
LF_TASK_HINT["lf5_petiole"] = "5_petiole"

for path in [GOLD_CSV, VOTES_DIR, UTILS]:
    if not path.exists():
        raise SystemExit("Khong thay " + str(path))

print("ROOT           :", ROOT)
print("GOLD_CSV       :", GOLD_CSV)
print("VOTES_DIR      :", VOTES_DIR)
print("OOF_RAW_CSV    :", OOF_RAW_CSV, "| ton tai:", OOF_RAW_CSV.exists())
print("REPORT_CSV     :", REPORT_CSV)
print("ALPHA_MIN      :", ALPHA_MIN)
print("COVERAGE_MIN   :", COVERAGE_MIN)
print("TAU_GRID       :", TAU_GRID)
print("TASKS          :", TASKS)

ROOT           : /sessions/serene-nice-babbage/mnt/coconut-iqa
GOLD_CSV       : /sessions/serene-nice-babbage/mnt/coconut-iqa/gold_seed/gold_seed_labels.csv
VOTES_DIR      : /sessions/serene-nice-babbage/mnt/coconut-iqa/labels/votes
OOF_RAW_CSV    : /sessions/serene-nice-babbage/mnt/coconut-iqa/labels/disease_clf/oof_predictions.csv | ton tai: True
REPORT_CSV     : /sessions/serene-nice-babbage/mnt/coconut-iqa/labels/validation/gold_seed_report.csv
ALPHA_MIN      : 0.5
COVERAGE_MIN   : 0.5
TAU_GRID       : [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
TASKS          : ['1_maturity_evaluation', '2_foliar_disease', '3_trunk_disease', '4_crown_disease', '5_petiole']


In [3]:
def is_lf6(lf_name):
    if lf_name is None:
        return False
    text = str(lf_name)
    if text.startswith("lf6"):
        return True
    return False


def gold_domain(gold_df, task):
    # Mien cua LF tren gold = cac anh gold co task_native == task (view goc cua tac vu).
    mask = gold_df["task_native"] == task
    return gold_df[mask]


def binary_prf(y_true, y_vote):
    prf = precision_recall_fscore_support(
        y_true,
        y_vote,
        labels=[1],
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    out = {}
    out["precision"] = prf[0]
    out["recall"] = prf[1]
    out["f1"] = prf[2]
    return out


def match_domain_votes(gold_df, votes_lf, task):
    # Ghep gold-domain voi phieu LF theo image_id (inner = khong abstain trong mien).
    domain = gold_domain(gold_df, task)
    sub = votes_lf[votes_lf["task"] == task]
    merged = domain.merge(
        sub,
        on="image_id",
        how="inner",
        suffixes=("", "_vote"),
    )
    return domain, merged


def validate_pair(gold_df, votes_lf, lf_name, task):
    domain, merged = match_domain_votes(gold_df, votes_lf, task)
    n_domain = len(domain)
    n_overlap = len(merged)
    row = {}
    row["lf"] = lf_name
    row["task"] = task
    row["n_domain"] = n_domain
    row["n_overlap"] = n_overlap
    row["coverage"] = ""
    row["alpha_hat"] = ""
    row["kappa"] = ""
    row["precision"] = ""
    row["recall"] = ""
    row["f1"] = ""
    row["tau_suggested"] = ""
    if n_domain == 0:
        row["coverage"] = 0.0
        row["decision"] = "bo qua (gold khong co task nay)"
        return row
    coverage = n_overlap / n_domain
    row["coverage"] = round(coverage, 4)
    if n_overlap == 0:
        row["decision"] = "loai (coverage 0)"
        return row
    y_true = merged[task].astype(int).tolist()
    y_vote = merged["vote"].astype(int).tolist()
    row["alpha_hat"] = round(accuracy_score(y_true, y_vote), 4)
    row["kappa"] = round(cohen_kappa_score(y_true, y_vote), 4)
    prf = binary_prf(y_true, y_vote)
    row["precision"] = round(prf["precision"], 4)
    row["recall"] = round(prf["recall"], 4)
    row["f1"] = round(prf["f1"], 4)
    keep = True
    if row["alpha_hat"] <= ALPHA_MIN:
        keep = False
    if coverage < COVERAGE_MIN:
        keep = False
    if keep:
        row["decision"] = "giu"
    else:
        row["decision"] = "loai (dua LF6 + gold seed)"
    return row


def per_source_rows(gold_df, votes_lf, lf_name, task):
    domain, merged = match_domain_votes(gold_df, votes_lf, task)
    out = []
    if len(merged) == 0:
        return out
    for src in sorted(merged["source"].unique()):
        part = merged[merged["source"] == src]
        y_true = part[task].astype(int).tolist()
        y_vote = part["vote"].astype(int).tolist()
        rec = {}
        rec["lf"] = lf_name
        rec["task"] = task
        rec["source"] = src
        rec["n"] = len(part)
        rec["alpha_hat"] = round(accuracy_score(y_true, y_vote), 4)
        prf = binary_prf(y_true, y_vote)
        rec["f1"] = round(prf["f1"], 4)
        out.append(rec)
    return out

In [4]:
def scan_tau(gold_df, merged, task, n_domain, tau_source):
    # Quet tau: coi phieu co confidence < tau la abstain, tinh lai coverage/alpha/F1.
    best = {}
    best["tau"] = ""
    best["f1"] = -1.0
    best["alpha_hat"] = ""
    best["coverage"] = ""
    best["tau_source"] = tau_source
    best["conf_min_observed"] = ""
    if len(merged) == 0:
        return best
    best["conf_min_observed"] = round(float(merged["confidence"].min()), 4)
    for tau in TAU_GRID:
        keep_mask = merged["confidence"] >= tau
        kept = merged[keep_mask]
        n_kept = len(kept)
        if n_kept == 0:
            continue
        coverage = n_kept / n_domain
        y_true = kept[task].astype(int).tolist()
        y_vote = kept["vote"].astype(int).tolist()
        prf = binary_prf(y_true, y_vote)
        f1 = prf["f1"]
        if f1 > best["f1"]:
            best["tau"] = tau
            best["f1"] = round(f1, 4)
            best["alpha_hat"] = round(accuracy_score(y_true, y_vote), 4)
            best["coverage"] = round(coverage, 4)
    return best


def tau_from_votes(gold_df, raw_df, task):
    # Nguon: file votes (da bo abstain luc sinh -> chi quet duoc tau >= tau luc sinh).
    domain = gold_domain(gold_df, task)
    n_domain = len(domain)
    sub = raw_df[raw_df["task"] == task].copy()
    sub = sub[sub["confidence"].notna()]
    merged = domain.merge(
        sub,
        on="image_id",
        how="inner",
        suffixes=("", "_vote"),
    )
    result = scan_tau(gold_df, merged, task, n_domain, "votes")
    return result


def tau_from_oof(gold_df, oof_df, task):
    # Nguon tho: oof_predictions (correctness = pred_class == gt_class) -> quet tron dai tau.
    domain = gold_domain(gold_df, task)
    n_domain = len(domain)
    sub = oof_df[oof_df["gt_task"] == task].copy()
    sub = sub[sub["pred_class"].notna()]
    sub = sub[sub["conf"].notna()]
    result = {}
    result["tau"] = ""
    result["f1"] = -1.0
    result["alpha_hat"] = ""
    result["coverage"] = ""
    result["tau_source"] = "oof"
    result["conf_min_observed"] = ""
    result["n_usable"] = len(sub)
    if len(sub) == 0:
        return result
    correct = sub["pred_class"] == sub["gt_class"]
    sub["vote"] = correct.astype(int)
    sub["confidence"] = sub["conf"]
    keep_cols = ["image_id", "vote", "confidence"]
    merged = domain.merge(
        sub[keep_cols],
        on="image_id",
        how="inner",
        suffixes=("", "_vote"),
    )
    scanned = scan_tau(gold_df, merged, task, n_domain, "oof")
    for key in scanned:
        result[key] = scanned[key]
    result["n_usable"] = len(merged)
    return result


def has_confidence(df):
    if "confidence" not in df.columns:
        return False
    if df["confidence"].notna().sum() == 0:
        return False
    return True

In [5]:
gold_df = pd.read_csv(GOLD_CSV)
long_df, wide_df = fuse_votes(VOTES_DIR)

oof_df = None
if OOF_RAW_CSV.exists():
    oof_df = pd.read_csv(OOF_RAW_CSV)

print("gold seed anh    :", len(gold_df))
print("phieu (long)     :", len(long_df))
print("cap (lf, task)   :")
print(long_df.groupby(["lf", "task"]).size().reset_index(name="n_phieu").to_string(index=False))

gold seed anh    : 300
phieu (long)     : 4736
cap (lf, task)   :
          lf                  task  n_phieu
lf1_maturity 1_maturity_evaluation      942
  lf2_foliar      2_foliar_disease     3794


In [6]:
report_rows = []
source_rows = []
tau_rows = []
notes = []

vote_files = sorted(VOTES_DIR.glob("lf*.csv"))
for vf in vote_files:
    raw = pd.read_csv(vf)
    stem = vf.stem

    if len(raw) == 0:
        hinted_task = LF_TASK_HINT.get(stem, "")
        row = {}
        row["lf"] = stem
        row["task"] = hinted_task
        row["n_domain"] = len(gold_domain(gold_df, hinted_task))
        row["n_overlap"] = 0
        row["coverage"] = 0.0
        row["alpha_hat"] = ""
        row["kappa"] = ""
        row["precision"] = ""
        row["recall"] = ""
        row["f1"] = ""
        row["tau_suggested"] = ""
        row["decision"] = "loai (0 phieu)"
        report_rows.append(row)
        notes.append(stem + ": 0 phieu (classifier chua train) -> coverage 0, bo qua alpha/kappa")
        continue

    lf_name = raw["lf"].iloc[0]

    if is_lf6(lf_name):
        row = {}
        row["lf"] = lf_name
        row["task"] = "(moi task)"
        row["n_domain"] = ""
        row["n_overlap"] = ""
        row["coverage"] = ""
        row["alpha_hat"] = ""
        row["kappa"] = ""
        row["precision"] = ""
        row["recall"] = ""
        row["f1"] = ""
        row["tau_suggested"] = ""
        row["decision"] = "loai khoi doi-chieu-theo-anh"
        report_rows.append(row)
        notes.append(lf_name + ": image_id tong hop (base__axis__delta) khong khop gold -> kiem dinh bang duong cong suy giam rieng")
        continue

    votes_lf = raw[["image_id", "task", "vote"]].copy()
    conf_ok = has_confidence(raw)

    for task in sorted(raw["task"].unique()):
        base = validate_pair(gold_df, votes_lf, lf_name, task)

        tau_pick = ""
        oof_result = None
        if oof_df is not None:
            oof_result = tau_from_oof(gold_df, oof_df, task)
        if oof_result is not None and oof_result["n_usable"] > 0:
            tau_rows.append(oof_result)
            tau_pick = oof_result["tau"]
        elif conf_ok:
            votes_result = tau_from_votes(gold_df, raw, task)
            tau_rows.append(votes_result)
            tau_pick = votes_result["tau"]

        base["tau_suggested"] = tau_pick
        report_rows.append(base)

        for rec in per_source_rows(gold_df, votes_lf, lf_name, task):
            source_rows.append(rec)

if oof_df is not None:
    usable = oof_df["pred_class"].notna().sum()
    if usable == 0:
        notes.append("oof_predictions.csv: pred_class rong (classifier chua train) -> khong quet duoc tau tron dai; dung confidence trong file votes")

In [7]:
report = pd.DataFrame(report_rows)
report = report[[
    "lf",
    "task",
    "n_domain",
    "n_overlap",
    "coverage",
    "alpha_hat",
    "kappa",
    "precision",
    "recall",
    "f1",
    "tau_suggested",
    "decision",
]]

print("=== BANG TONG HOP (lf, task) ===")
print(report.to_string(index=False))

print()
print("=== TACH THEO SOURCE ===")
if source_rows:
    print(pd.DataFrame(source_rows).to_string(index=False))
else:
    print("(khong co)")

print()
print("=== CALIBRATE TAU ===")
if tau_rows:
    tau_df = pd.DataFrame(tau_rows)
    print(tau_df.to_string(index=False))
else:
    print("(khong co LF nao co confidence de quet tau)")

print()
print("=== GHI CHU ===")
for line in notes:
    print("-", line)

REPORT_CSV.parent.mkdir(parents=True, exist_ok=True)
report.to_csv(REPORT_CSV, index=False)
print()
print("ghi:", REPORT_CSV, "|", len(report), "dong")

=== BANG TONG HOP (lf, task) ===
          lf                  task  n_domain  n_overlap  coverage alpha_hat   kappa precision  recall      f1 tau_suggested       decision
lf1_maturity 1_maturity_evaluation        60         60       1.0    0.7833 -0.0598    0.9592  0.8103  0.8785          0.85            giu
  lf2_foliar      2_foliar_disease        60         60       1.0    0.9833     0.0    0.9833     1.0  0.9916                          giu
   lf3_trunk       3_trunk_disease        60          0       0.0                                                           loai (0 phieu)

=== TACH THEO SOURCE ===
          lf                  task                 source  n  alpha_hat     f1
lf1_maturity 1_maturity_evaluation  coconut-veirf-v5/test  7     0.7143 0.8333
lf1_maturity 1_maturity_evaluation coconut-veirf-v5/train 41     0.7561 0.8611
lf1_maturity 1_maturity_evaluation coconut-veirf-v5/valid 12     0.9167 0.9565
  lf2_foliar      2_foliar_disease disease/Gray Leaf Spot 32     1.00

## Giới hạn

`fuse_votes` chỉ giữ `lf, image_id, task, vote` (bỏ `confidence` theo thiết kế) — bước calibrate tau đọc lại file votes gốc để lấy `confidence`.

File votes đã bỏ dòng abstain lúc sinh, nên chỉ quét được `tau >= tau lúc sinh`. Muốn quét trọn dải tau phải đọc điểm thô `labels/disease_clf/oof_predictions.csv`; notebook tự ưu tiên nguồn thô khi có dự đoán, ngược lại lùi về `confidence` trong file votes.

LF6 (degradation) dùng `image_id` tổng hợp `base__axis__delta`, không khớp gold theo ảnh → loại khỏi đối chiếu-theo-ảnh; kiểm định bằng đường cong suy giảm riêng. Xem thêm tại `docs/LF3_Methodology.md`.

Cổng go/no-go: xem thêm tại `docs/LF1_Methodology.md`, `docs/LF3_Methodology.md`.